In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
    session_id INT,
    zone_id STRING,
    vehicle_type STRING,
    duration_hours INT,
    parking_fee INT,
    status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
    zone_id STRING,
    zone_name STRING,
    city STRING,
    capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TABLE incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
);

num_affected_rows,num_inserted_rows


Section A — SQL and DataFrames (20 marks)

1. Display all records from parking_sessions_source . Retrieve only session_id ,
vehicle_type , and parking_fee .
2. Find all completed parking sessions where the fee is greater than ₹100.
3. Calculate the total parking fees and number of completed sessions for each vehicle type using
GROUP BY .
4. Using PySpark, read parking_sessions_source into a DataFrame, add a column named
fee_category ( High for fees of ₹200 or more, otherwise Low ), and display the result

In [0]:
%sql
select * from parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


In [0]:
%sql
select session_id,vehicle_type,parking_fee from parking_sessions_work;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
select * from parking_sessions_work where status='Completed' and parking_fee>100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
select vehicle_type,sum(parking_fee) as total_parking_fee, count(*) as total_sessions from parking_sessions_work group by vehicle_type;

vehicle_type,total_parking_fee,total_sessions
Car,1750,8
Bike,260,6
SUV,1320,6


In [0]:
%python
df=spark.table("parking_sessions_work")
df.show(5)


+----------+-------+------------+--------------+-----------+---------+
|session_id|zone_id|vehicle_type|duration_hours|parking_fee|   status|
+----------+-------+------------+--------------+-----------+---------+
|       101|    Z01|         Car|             3|        150|Completed|
|       102|    Z02|        Bike|             2|         40|Completed|
|       103|    Z03|         Car|             5|        250|Completed|
|       104|    Z01|         SUV|             4|        240|   Active|
|       105|    Z04|        Bike|             1|         20|Completed|
+----------+-------+------------+--------------+-----------+---------+
only showing top 5 rows


In [0]:
%python
from pyspark.sql.functions import when,col
df=df.withColumn("fee_category",when(col("parking_fee")>200,"high").otherwise ("low"))
df.show()

+----------+-------+------------+--------------+-----------+---------+------------+
|session_id|zone_id|vehicle_type|duration_hours|parking_fee|   status|fee_category|
+----------+-------+------------+--------------+-----------+---------+------------+
|       101|    Z01|         Car|             3|        150|Completed|         low|
|       102|    Z02|        Bike|             2|         40|Completed|         low|
|       103|    Z03|         Car|             5|        250|Completed|        high|
|       104|    Z01|         SUV|             4|        240|   Active|        high|
|       105|    Z04|        Bike|             1|         20|Completed|         low|
|       106|    Z02|         Car|             6|        300|Completed|        high|
|       107|    Z03|         SUV|             2|        120|Cancelled|         low|
|       108|    Z05|         Car|             3|        150|Completed|         low|
|       109|    Z01|        Bike|             4|         80|Completed|      

Section B — Delta Tables and CRUD (20 marks)

5. Create a new managed Delta table named parking_summary using CTAS, containing only
completed sessions.
6. Insert a new record into parking_sessions_work with session ID 123, zone Z01, vehicle type
Car, duration 3 hours, fee ₹150, and status Active.
7. Update session 104 to Completed and increase its parking fee to ₹280.
8. Delete all Cancelled sessions from parking_sessions_work . Display the remaining records

In [0]:
create table manage_delta_table as select * from parking_sessions_work where status='Completed';

num_affected_rows,num_inserted_rows


In [0]:
insert into parking_sessions_work values(123,'Z01','Car',3,150,'Active');

num_affected_rows,num_inserted_rows
1,1


In [0]:
update parking_sessions_work set status='Completed',parking_fee=280  where session_id=104;

num_affected_rows
1


In [0]:
delete from parking_sessions_work where status='Cancelled';

num_affected_rows
2


In [0]:
select * from parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


Section C — MERGE and Delta Operations (20 marks)

9. Display incoming_parking_sessions and identify which records already exist in
parking_sessions_work .
10. Perform a MERGE operation using session_id to update existing records and insert new
records.
11. Use DESCRIBE HISTORY to display the transaction history of parking_sessions_work .
12. Use Time Travel to retrieve the table version immediately before the MERGE operation. Compare
session 102 before and after MERGE

In [0]:
select * from incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
select * from parking_sessions_work as p
inner join incoming_parking_sessions as i on 
p.session_id=i.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,2,40,Completed,102,Z02,Bike,3,60,Completed
104,Z01,SUV,4,280,Completed,104,Z01,SUV,5,300,Completed


In [0]:
MERGE INTO parking_sessions_work AS p
USING incoming_parking_sessions AS i
ON p.session_id = i.session_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
describe history parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-11-12T06:29:34.000Z,74600853328676,luxya2026@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1407267923463881),57250cf2-d300-4b41-b8f3-8c5e583235a9,1112-060429-cscem5rv-v2n,11,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9846, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
11,2026-11-12T06:29:31.000Z,74600853328676,luxya2026@gmail.com,MERGE,"Map(predicate -> [""(session_id#13925 = session_id#13907)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1407267923463881),57250cf2-d300-4b41-b8f3-8c5e583235a9,1112-060429-cscem5rv-v2n,10,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7568, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4008, materializeSourceTimeMs -> 368, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1216, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2317)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
10,2026-11-12T06:19:30.000Z,74600853328676,luxya2026@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1407267923463881),7beac52a-0ed9-42f9-9dd0-03e22e986f5b,1112-060429-cscem5rv-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6201, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-12T06:19:28.000Z,74600853328676,luxya2026@gmail.com,DELETE,"Map(predicate -> [""(status#12195 = Cancelled)""])",null,List(1407267923463881),7beac52a-0ed9-42f9-9dd0-03e22e986f5b,1112-060429-cscem5rv-v2n,8,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1131, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 623, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 507)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-12T06:18:47.000Z,74600853328676,luxya2026@gmail.com,UPDATE,"Map(predicate -> [""(session_id#11719 = 104)""])",null,List(1407267923463881),dcdf0fcf-c2b1-420a-bd1d-4b08ce7aac63,1112-060429-cscem5rv-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1932, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1955, numDeletionVectorsUpdated -> 0, scanTimeMs -> 726, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 1228)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-12T06:18:18.000Z,74600853328676,luxya2026@gmail.com,UPDATE,"Map(predicate -> [""(session_id#11224 = 104)""])",null,List(1407267923463881),9cc8c526-3d5e-4787-ba4b-4060da87a2c2,1112-060429-cscem5rv-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows

In [0]:
select * from parking_sessions_work version as of 10 where session_id=102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,2,40,Completed


In [0]:
select * from parking_sessions_work version as of 11 where session_id=102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


Section D — Tables, Views and Optimization (20 marks)

13. Create a permanent view named completed_parking_view containing only completed
sessions from parking_sessions_work .
14. Create a temporary view containing parking sessions from zone Z01 and a global temporary view
containing sessions with parking fees above ₹200. Query both views.
15. Join parking_sessions_work with parking_zones and display the zone name, city,
vehicle type, and parking fee.
16. Run DESCRIBE DETAIL on parking_sessions_work , execute OPTIMIZE , and compare
the file count before and after optimization. Explain why a small dataset might show no significant
change

In [0]:
%sql
CREATE OR REPLACE  view completed_parking_sessions_work AS
SELECT *
FROM parking_sessions_work where status='Completed';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_parking_view AS
SELECT *
FROM parking_sessions_work
WHERE zone_id = 'Z01';

In [0]:
%sql
SELECT * FROM temp_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_parking_view AS
SELECT *
FROM parking_sessions_work
WHERE parking_fee > 200;

In [0]:
%sql
SELECT * FROM global_temp.high_fee_parking_view;

In [0]:
select z.zone_name,z.city,p.vehicle_type,p.parking_fee  from parking_sessions_work p  inner join parking_zones z on 
p.zone_id=z.zone_id; 


zone_name,city,vehicle_type,parking_fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cdc146db-f7ce-48d4-a7c5-ba414e804ca1,workspace.default.parking_sessions_work,null,,2026-11-12T05:56:04.177Z,2026-11-12T06:29:34.000Z,List(),List(),1,2287,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.checkpoint.writeStatsAsJson -> false)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
optimize parking_sessions_work;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1794465665176, 1794465665983, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


In [0]:
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cdc146db-f7ce-48d4-a7c5-ba414e804ca1,workspace.default.parking_sessions_work,null,,2026-11-12T05:56:04.177Z,2026-11-12T06:29:34.000Z,List(),List(),1,2287,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.checkpoint.writeStatsAsJson -> false)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Why might a small dataset show no significant change after OPTIMIZE?

1. Already fewer files: A small dataset may already be stored in just one or two files.
2. Nothing much to combine: OPTIMIZE works by combining smaller files into larger files. If there are very few files, there is little work to do.

Section E — Structured Streaming and DLT (20 marks)

17. Using PySpark's built-in rate source, generate five streaming records per second and display
the stream.
18. Apply a streaming transformation that creates an amount column from value and filters
records with an amount greater than ₹300.
19. Create a Lakeflow/DLT SQL pipeline with a Bronze streaming table reading from
parking_sessions_source and a Silver streaming table retaining only completed sessions.
20. Create a Gold materialized view that calculates total completed sessions and total parking
revenue by zone. Run the pipeline and verify the Gold output

In [0]:
%python
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

display(rate_df)

In [0]:
%python
from pyspark.sql.functions import col

amount_df = rate_df.withColumn("amount", col("value")) \
    .filter(col("amount") > 300)

display(amount_df)

In [0]:
CREATE OR REFRESH STREAMING TABLE parking_sessions_bronze
AS
SELECT *
FROM STREAM(parking_sessions_source);

message
The operation was successfully executed.


In [0]:
CREATE OR REFRESH STREAMING TABLE parking_sessions_silver
AS
SELECT *
FROM STREAM(parking_sessions_bronze)
WHERE status = 'Completed';

message
The operation was successfully executed.


In [0]:
CREATE OR REFRESH MATERIALIZED VIEW parking_zone_gold
AS
SELECT
    zone_id,
    COUNT(*) AS total_completed_sessions,
    SUM(parking_fee) AS total_parking_revenue
FROM parking_sessions_silver
GROUP BY zone_id;

result
The operation was successfully executed.


In [0]:
%sql
SELECT *
FROM parking_zone_gold
ORDER BY zone_id;

zone_id,total_completed_sessions,total_parking_revenue
Z01,3,410
Z02,4,800
Z03,3,620
Z04,2,270
Z05,2,350
